# 🔬 Module 22 Lab: Fine-tuning & Model Specialization

**Mục tiêu:** Simulate toàn bộ fine-tuning pipeline cho LoanBot — từ data preparation đến evaluation và deployment.

**Không cần GPU:** Lab này dùng mock models và simulated training để focus vào concepts và pipeline design.

**Sections:**
1. ROI Calculator & Decision Framework
2. LoRA Config Explorer
3. Synthetic Data Generator & Quality Filter
4. Fine-tuning Simulator với Early Stopping
5. 4-Dimension Evaluation Suite
6. Bias Detection & Fairness Audit
7. Model Registry & Canary Router
8. Practice: Complete Pipeline Orchestrator

In [ ]:
# Setup — no external dependencies for core exercises
import json, math, random, hashlib, time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
import statistics

random.seed(42)
print('✅ Setup complete — all imports OK')

## Section 1: ROI Calculator & Decision Framework

Trước khi fine-tune, phải chứng minh được ROI. FinTech Corp cần tính break-even point.

In [ ]:
@dataclass
class FineTuningROI:
    monthly_requests: int
    base_model_cost_per_req: float   # USD — includes token costs for long system prompt
    ft_model_cost_per_req: float     # USD — fine-tuned model: shorter prompt + potentially smaller model
    training_cost: float             # USD — one-time: compute + data preparation
    model_lifespan_months: int       # how long fine-tuned model stays relevant before retraining

    def monthly_savings(self) -> float:
        return (self.base_model_cost_per_req - self.ft_model_cost_per_req) * self.monthly_requests

    def break_even_months(self) -> float:
        ms = self.monthly_savings()
        if ms <= 0:
            return float('inf')
        return self.training_cost / ms

    def total_roi(self) -> float:
        """Total net savings over model lifespan."""
        return self.monthly_savings() * self.model_lifespan_months - self.training_cost

    def recommend(self) -> str:
        bem = self.break_even_months()
        roi = self.total_roi()
        if bem > self.model_lifespan_months:
            return f'❌ Do NOT fine-tune — break-even {bem:.1f}m > lifespan {self.model_lifespan_months}m'
        elif roi > 0:
            return f'✅ Fine-tune — break-even {bem:.1f}m, total ROI ${roi:,.0f} over {self.model_lifespan_months}m'
        else:
            return f'⚠️  Borderline — ROI ${roi:,.0f}, reconsider'


# LoanBot realistic scenario
loanbot_roi = FineTuningROI(
    monthly_requests=50_000,
    base_model_cost_per_req=0.008,   # Claude Sonnet with 3K token system prompt
    ft_model_cost_per_req=0.002,     # Fine-tuned Llama-3-8B: cheaper hosting + no long prompt
    training_cost=2_000,             # GPU compute + data prep time
    model_lifespan_months=18,
)

print('=== LoanBot Fine-tuning ROI Analysis ===')
print(f'Monthly savings: ${loanbot_roi.monthly_savings():,.0f}')
print(f'Break-even: {loanbot_roi.break_even_months():.1f} months')
print(f'Total ROI over {loanbot_roi.model_lifespan_months}m: ${loanbot_roi.total_roi():,.0f}')
print(f'Recommendation: {loanbot_roi.recommend()}')

## Section 2: LoRA Config Explorer

LoRA có nhiều hyperparameters. Explorer này tính số trainable params và VRAM estimate cho các configs khác nhau.

In [ ]:
@dataclass
class LoRAConfig:
    r: int                         # rank
    alpha: float                   # scaling factor
    target_modules: List[str]      # e.g. ['q_proj', 'v_proj']
    dropout: float
    base_model_params_B: float     # base model size in billions
    hidden_dim: int                # d_model: 4096 for 7B, 8192 for 70B

    def trainable_params(self) -> int:
        """Each LoRA module: B (hidden_dim × r) + A (r × hidden_dim)"""
        params_per_module = 2 * self.hidden_dim * self.r
        return params_per_module * len(self.target_modules)

    def trainable_pct(self) -> float:
        total = self.base_model_params_B * 1e9
        return self.trainable_params() / total * 100

    def scaling_factor(self) -> float:
        return self.alpha / self.r

    def vram_estimate_gb(self, quantization_bits: int = 16) -> float:
        """Rough VRAM: model + optimizer states for trainable params."""
        bytes_per_param = quantization_bits / 8
        model_gb = self.base_model_params_B * 1e9 * bytes_per_param / 1e9
        # Optimizer (Adam): 3 states × fp32 for trainable params
        optim_gb = self.trainable_params() * 3 * 4 / 1e9
        # Activations: rough estimate
        act_gb = 2.0
        return model_gb + optim_gb + act_gb

    def report(self, name: str):
        print(f'\n--- {name} ---')
        print(f'  r={self.r}, alpha={self.alpha}, scaling={self.scaling_factor():.2f}')
        print(f'  Target modules: {self.target_modules}')
        print(f'  Trainable params: {self.trainable_params():,} ({self.trainable_pct():.3f}%)')
        print(f'  VRAM (fp16): {self.vram_estimate_gb(16):.1f} GB')
        print(f'  VRAM (4-bit QLoRA): {self.vram_estimate_gb(4):.1f} GB')


# Compare 3 configs for 7B model
configs = [
    LoRAConfig(r=8,  alpha=16,  target_modules=['q_proj','v_proj'],
               dropout=0.05, base_model_params_B=7.0, hidden_dim=4096),
    LoRAConfig(r=16, alpha=32,  target_modules=['q_proj','k_proj','v_proj','o_proj'],
               dropout=0.05, base_model_params_B=7.0, hidden_dim=4096),
    LoRAConfig(r=64, alpha=128, target_modules=['q_proj','k_proj','v_proj','o_proj'],
               dropout=0.0,  base_model_params_B=7.0, hidden_dim=4096),
]
names = ['Config A: r=8 (minimal)', 'Config B: r=16 (recommended)', 'Config C: r=64 (maximal)']

print('=== LoRA Config Comparison for LoanBot (7B model) ===')
for cfg, name in zip(configs, names):
    cfg.report(name)

## Section 3: Synthetic Data Generator & Quality Filter

Generate anonymized training examples từ production data.

In [ ]:
@dataclass
class LoanApplication:
    customer_id: str
    credit_score: int
    dti_ratio: float
    employment_months: int
    loan_amount_vnd: int
    province: str
    # PII fields (will be removed)
    full_name: str = ''
    cccd: str = ''
    phone: str = ''

@dataclass
class ExpertDecision:
    decision: str       # APPROVE | REJECT | REVIEW
    confidence: float
    explanation: str
    regulation_ref: str

@dataclass
class TrainingExample:
    instruction: str
    anonymized_app: dict
    output: dict
    quality_score: float
    source: str = 'synthetic'


def anonymize(app: LoanApplication) -> dict:
    """Remove all PII before training pipeline."""
    pseudo_id = 'KH_' + hashlib.md5(app.customer_id.encode()).hexdigest()[:6].upper()
    return {
        'customer_id': pseudo_id,
        'credit_score': app.credit_score,
        'dti_ratio': round(app.dti_ratio, 2),
        'employment_months': app.employment_months,
        'loan_amount_vnd': round(app.loan_amount_vnd, -7),  # round to 10M
        'province': app.province,
        # NO: full_name, cccd, phone
    }


def mock_generate_example(app: LoanApplication, decision: ExpertDecision) -> TrainingExample:
    """Simulate Claude-generated synthetic training example."""
    anon = anonymize(app)

    decision_logic = {
        'APPROVE': f'Hồ sơ {anon["customer_id"]} đạt tiêu chuẩn thẩm định: credit score {anon["credit_score"]} vượt ngưỡng tối thiểu 600, DTI {anon["dti_ratio"]} trong giới hạn 45% theo TT39/2016.',
        'REJECT':  f'Từ chối hồ sơ {anon["customer_id"]}: credit score {anon["credit_score"]} dưới ngưỡng tối thiểu 500 theo chính sách FinTech Corp. Khuyến nghị: cải thiện credit profile trong 6 tháng.',
        'REVIEW':  f'Hồ sơ {anon["customer_id"]} cần xem xét thêm: DTI {anon["dti_ratio"]} ở mức borderline (40-45%). Yêu cầu chứng minh thu nhập bổ sung theo Điều 15 NĐ13/2023.',
    }

    instruction = f'Thẩm định hồ sơ vay với credit score {anon["credit_score"]}, DTI {anon["dti_ratio"]}, thâm niên {anon["employment_months"]} tháng'
    output = {
        'decision': decision.decision,
        'confidence': decision.confidence,
        'explanation': decision_logic[decision.decision],
        'regulation_refs': [decision.regulation_ref],
    }

    # Quality score: penalize short explanations, reward regulation refs
    quality = 0.5
    quality += 0.2 if len(output['explanation']) > 80 else 0
    quality += 0.15 if output['regulation_refs'] else 0
    quality += 0.1 if decision.confidence >= 0.80 else 0
    quality += 0.05 if len(instruction) > 20 else 0

    return TrainingExample(instruction=instruction, anonymized_app=anon, output=output, quality_score=round(quality, 2))


def filter_examples(examples: List[TrainingExample], threshold: float = 0.70) -> List[TrainingExample]:
    """Apply quality + dedup + length filters."""
    seen = set()
    filtered = []
    for ex in examples:
        if ex.quality_score < threshold: continue
        if len(ex.instruction) < 20 or len(ex.instruction) > 300: continue
        h = hashlib.md5(ex.instruction.encode()).hexdigest()
        if h in seen: continue
        seen.add(h)
        filtered.append(ex)
    return filtered


# Generate 20 mock production cases
PROVINCES = ['Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Cần Thơ', 'Nghệ An', 'Thanh Hóa', 'Quảng Ngãi']

def random_case(i: int):
    score = random.choice([350, 450, 520, 580, 620, 660, 700, 720, 750])
    dti = round(random.uniform(0.20, 0.55), 2)
    emp = random.choice([3, 6, 12, 24, 36, 60])
    app = LoanApplication(
        customer_id=f'PROD-2024-{i:04d}',
        credit_score=score, dti_ratio=dti,
        employment_months=emp,
        loan_amount_vnd=random.choice([100_000_000, 200_000_000, 500_000_000]),
        province=random.choice(PROVINCES),
        full_name=f'Nguyen Van X{i}',  # PII — will be removed
        cccd='0' + str(random.randint(10**11, 10**12-1)),
    )
    if score >= 650 and dti < 0.42: dec, conf = 'APPROVE', 0.92
    elif score < 500: dec, conf = 'REJECT', 0.95
    else: dec, conf = 'REVIEW', 0.75
    ref = 'TT39/2016 Điều 9' if dec != 'REVIEW' else 'NĐ13/2023 Điều 15'
    decision = ExpertDecision(dec, conf, '', ref)
    return app, decision

cases = [random_case(i) for i in range(20)]
raw_examples = [mock_generate_example(app, dec) for app, dec in cases]
filtered = filter_examples(raw_examples, threshold=0.70)

print(f'Generated: {len(raw_examples)} examples')
print(f'After quality filter (≥0.70): {len(filtered)} examples')
print(f'\nSample example:')
ex = filtered[0]
print(f'  Instruction: {ex.instruction}')
print(f'  Decision: {ex.output["decision"]} (conf: {ex.output["confidence"]})')
print(f'  Quality score: {ex.quality_score}')
print(f'  PII check - full_name present: {"full_name" in ex.anonymized_app}')
print(f'  PII check - cccd present: {"cccd" in ex.anonymized_app}')

## Section 4: Fine-tuning Simulator với Early Stopping

Simulate training curves để visualize overfitting và early stopping.

In [ ]:
def simulate_training(
    n_epochs: int,
    train_examples: int,
    rank: int,
    target_accuracy: float = 0.97,
    overfitting_start: int = 3,
) -> List[Dict]:
    """Simulate training curves for LoanBot fine-tuning."""
    history = []
    base_acc = 0.82
    noise = 0.015

    for epoch in range(1, n_epochs + 1):
        # Train accuracy: monotonically increases (with noise)
        train_acc = min(0.999, base_acc + (target_accuracy - base_acc) * (1 - math.exp(-epoch * 0.7)) + random.gauss(0, noise/2))

        # Val accuracy: peaks then decreases (overfitting)
        val_peak = target_accuracy - 0.01
        if epoch <= overfitting_start:
            val_acc = min(val_peak, base_acc + (val_peak - base_acc) * (1 - math.exp(-epoch * 0.6)) + random.gauss(0, noise))
        else:
            overfit_penalty = 0.015 * (epoch - overfitting_start)
            val_acc = max(0.75, val_peak - overfit_penalty + random.gauss(0, noise))

        # Loss (inversely related)
        train_loss = max(0.01, 2.5 * (1 - train_acc))
        val_loss   = max(0.01, 2.5 * (1 - val_acc) + max(0, (epoch - overfitting_start) * 0.05))

        history.append({
            'epoch': epoch,
            'train_acc': round(train_acc, 4),
            'val_acc': round(val_acc, 4),
            'train_loss': round(train_loss, 4),
            'val_loss': round(val_loss, 4),
        })
    return history


class EarlyStopping:
    def __init__(self, patience: int = 2, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_val_acc = 0.0
        self.best_epoch = 0
        self.counter = 0

    def step(self, epoch: int, val_acc: float) -> bool:
        """Returns True if training should stop."""
        if val_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = val_acc
            self.best_epoch = epoch
            self.counter = 0
            return False  # continue
        else:
            self.counter += 1
            return self.counter >= self.patience  # stop if patience exceeded


history = simulate_training(n_epochs=8, train_examples=7200, rank=16)
stopper = EarlyStopping(patience=2)

print('=== LoanBot Fine-tuning Simulation ===')
print(f'{"Epoch":<8}{"Train Acc":<12}{"Val Acc":<12}{"Val Loss":<12}{"Status"}')
print('-' * 55)

stopped_at = None
for h in history:
    should_stop = stopper.step(h['epoch'], h['val_acc'])
    status = '🏆 BEST' if stopper.best_epoch == h['epoch'] else ('🛑 STOP' if should_stop else '')
    print(f'{h["epoch"]:<8}{h["train_acc"]:<12.4f}{h["val_acc"]:<12.4f}{h["val_loss"]:<12.4f}{status}')
    if should_stop and stopped_at is None:
        stopped_at = h['epoch']
        print(f'\n⚡ Early stopping triggered at epoch {stopped_at}')
        print(f'   Best checkpoint: epoch {stopper.best_epoch} with val_acc={stopper.best_val_acc:.4f}')
        break

if stopped_at is None:
    print(f'\n✅ Training completed. Best epoch: {stopper.best_epoch}, val_acc={stopper.best_val_acc:.4f}')

## Section 5: 4-Dimension Evaluation Suite

Evaluate fine-tuned model trên 4 dimensions: domain accuracy, JSON compliance, general capability, latency.

In [ ]:
class MockBaseModel:
    """Simulates Claude base model + long system prompt."""
    name = 'claude-base'

    def predict(self, app: dict) -> dict:
        score = app.get('credit_score', 600)
        dti = app.get('dti_ratio', 0.4)
        # Slightly lenient on borderline cases
        if score >= 650 and dti < 0.45:
            dec = 'APPROVE'
        elif score < 500:
            dec = 'REJECT'
        else:
            dec = 'REVIEW'
        time.sleep(random.uniform(0.003, 0.007))  # simulate latency
        return {'decision': dec, 'confidence': 0.88, 'explanation': 'Thẩm định theo quy trình chuẩn của FinTech Corp. ' * 8}


class MockFineTunedModel:
    """Simulates fine-tuned LLaMA-3-8B with LoRA."""
    name = 'loanbot-ft-v3.6'

    def predict(self, app: dict) -> dict:
        score = app.get('credit_score', 600)
        dti = app.get('dti_ratio', 0.4)
        # Stricter on borderline — reflects training data
        if score >= 650 and dti < 0.42:
            dec = 'APPROVE'
        elif score < 500 or (score < 580 and dti > 0.42):
            dec = 'REJECT'
        else:
            dec = 'REVIEW'
        time.sleep(random.uniform(0.001, 0.003))  # faster — no long prompt
        return {
            'decision': dec,
            'confidence': 0.94,
            'explanation': f'KH có credit score {score}, DTI {dti} — đánh giá theo TT39/2016 Điều 9 và chính sách FinTech Corp.',
            'regulation_refs': ['TT39/2016'],
        }


# Test dataset
TEST_APPS = [
    {'credit_score': 720, 'dti_ratio': 0.30, 'expert': 'APPROVE'},
    {'credit_score': 680, 'dti_ratio': 0.38, 'expert': 'APPROVE'},
    {'credit_score': 450, 'dti_ratio': 0.50, 'expert': 'REJECT'},
    {'credit_score': 380, 'dti_ratio': 0.60, 'expert': 'REJECT'},
    {'credit_score': 610, 'dti_ratio': 0.44, 'expert': 'REVIEW'},
    {'credit_score': 590, 'dti_ratio': 0.41, 'expert': 'REVIEW'},
    {'credit_score': 660, 'dti_ratio': 0.40, 'expert': 'APPROVE'},
    {'credit_score': 520, 'dti_ratio': 0.35, 'expert': 'REVIEW'},
]

GENERAL_TASKS = [
    ('Dịch sang tiếng Anh: Xin chào', lambda r: 'hello' in r.lower()),
    ('234 nhân 17 bằng bao nhiêu?', lambda r: '3978' in r),
    ('Liệt kê: a, b, c dưới dạng bullet list', lambda r: '-' in r or '•' in r or '*' in r),
]


def evaluate_model(model, label: str):
    print(f'\n=== Evaluating: {label} ===')

    # Dim 1: Domain accuracy
    correct = sum(1 for app in TEST_APPS if model.predict(app)['decision'] == app['expert'])
    acc = correct / len(TEST_APPS)
    print(f'  📊 Domain Accuracy:     {acc:.1%} ({correct}/{len(TEST_APPS)})  {"✅" if acc >= 0.90 else "❌"}')

    # Dim 2: JSON compliance
    valid = 0
    for app in TEST_APPS:
        result = model.predict(app)
        if {'decision', 'confidence', 'explanation'}.issubset(result.keys()):
            if result['decision'] in {'APPROVE', 'REJECT', 'REVIEW'}:
                valid += 1
    json_ok = valid / len(TEST_APPS)
    print(f'  📋 JSON Compliance:     {json_ok:.1%} ({valid}/{len(TEST_APPS)})  {"✅" if json_ok >= 0.99 else "❌"}')

    # Dim 3: General capability (mock)
    GENERAL_RESPONSES = {
        'claude-base': ['Hello! How can I help?', '3978', '- a\n- b\n- c'],
        'loanbot-ft-v3.6': ['hello', '3978', '• a\n• b\n• c'],
    }
    responses = GENERAL_RESPONSES.get(model.name, ['', '', ''])
    gen_pass = sum(1 for (_, check), resp in zip(GENERAL_TASKS, responses) if check(resp))
    gen_score = gen_pass / len(GENERAL_TASKS)
    print(f'  🧠 General Capability:  {gen_score:.1%} ({gen_pass}/{len(GENERAL_TASKS)}) {"✅" if gen_score >= 0.85 else "❌"}')

    # Dim 4: Latency
    times = []
    for app in TEST_APPS:
        t = time.time()
        model.predict(app)
        times.append(time.time() - t)
    p50 = sorted(times)[len(times)//2]
    print(f'  ⚡ Latency p50:         {p50*1000:.1f} ms  {"✅" if p50 < 0.02 else "✅" if p50 < 0.05 else "⚠️"}')


evaluate_model(MockBaseModel(), 'Base Model (Claude + long prompt)')
evaluate_model(MockFineTunedModel(), 'Fine-tuned Model (LLaMA-3-8B + LoRA)')

## Section 6: Bias Detection & Fairness Audit

Fine-tuning có thể amplify historical bias. LoanBot cần pass disparate impact analysis.

In [ ]:
def disparate_impact_analysis(model, applications_by_group: Dict[str, List[dict]]) -> Dict:
    """Check if rejection rate differs significantly across groups.
    
    4/5 Rule: if rejection_rate(group_A) / rejection_rate(group_B) < 0.8, disparate impact.
    """
    results = {}
    for group, apps in applications_by_group.items():
        decisions = [model.predict(app)['decision'] for app in apps]
        reject_rate = decisions.count('REJECT') / len(decisions)
        approve_rate = decisions.count('APPROVE') / len(decisions)
        results[group] = {'reject_rate': reject_rate, 'approve_rate': approve_rate, 'n': len(decisions)}

    # Find group with lowest rejection rate (most favorable)
    best_group = min(results, key=lambda g: results[g]['reject_rate'])
    best_approve = results[best_group]['approve_rate']

    print('=== Disparate Impact Analysis ===')
    print(f'{"Group":<15}{"Approve%":<12}{"Reject%":<12}{"N":<8}{"4/5 Ratio":<12}{"Flagged?"}')
    print('-' * 65)

    flags = []
    for group, r in results.items():
        ratio = r['approve_rate'] / best_approve if best_approve > 0 else 1.0
        flagged = ratio < 0.80  # 4/5 rule
        if flagged: flags.append(group)
        print(f'{group:<15}{r["approve_rate"]:<12.1%}{r["reject_rate"]:<12.1%}{r["n"]:<8}{ratio:<12.2f}{"⚠️ FLAG" if flagged else "✅ OK"}')

    if flags:
        print(f'\n⚠️  Disparate impact detected for: {flags}')
        print('   Recommendation: collect more training data from flagged groups, apply fairness constraints')
    else:
        print('\n✅ No significant disparate impact detected')
    return {'flags': flags, 'results': results}


# Generate applications with similar credit quality across provinces
def gen_apps_for_group(n: int, score_range=(600, 720), dti_range=(0.28, 0.45)) -> List[dict]:
    return [{
        'credit_score': random.randint(*score_range),
        'dti_ratio': round(random.uniform(*dti_range), 2)
    } for _ in range(n)]

# Simulate historical bias: urban areas have better representation in training data
class BiasedFineTunedModel(MockFineTunedModel):
    """Model with geographic bias amplified from historical training data."""
    name = 'loanbot-ft-biased'

    def predict(self, app: dict) -> dict:
        result = super().predict(app)
        # Simulate bias: rural applicants get stricter threshold (historical pattern)
        if app.get('rural', False) and result['decision'] == 'APPROVE':
            if random.random() < 0.35:  # 35% of rural APPROVEs become REVIEW
                result['decision'] = 'REVIEW'
        return result


def gen_apps_group(n, is_rural=False):
    apps = gen_apps_for_group(n)
    for a in apps: a['rural'] = is_rural
    return apps

groups = {
    'Hà Nội/TP.HCM': gen_apps_group(50, is_rural=False),
    'Đà Nẵng':        gen_apps_group(50, is_rural=False),
    'Nghệ An':        gen_apps_group(50, is_rural=True),
    'Quảng Ngãi':     gen_apps_group(50, is_rural=True),
}

print('--- Unbiased fine-tuned model ---')
disparate_impact_analysis(MockFineTunedModel(), groups)
print()
print('--- Biased fine-tuned model (historical bias amplified) ---')
disparate_impact_analysis(BiasedFineTunedModel(), groups)

## Section 7: Model Registry & Canary Router

In [ ]:
@dataclass
class ModelVersion:
    version_id: str
    model_name: str
    status: str    # shadow | canary | production | retired
    traffic_pct: float
    accuracy: Optional[float] = None
    p50_ms: Optional[float] = None


class ModelRegistry:
    def __init__(self):
        self.versions: Dict[str, ModelVersion] = {}

    def register(self, v: ModelVersion):
        self.versions[v.version_id] = v
        print(f'📦 Registered {v.version_id} [{v.status}] traffic={v.traffic_pct:.0%}')

    def route(self, request_id: str) -> ModelVersion:
        """Deterministic routing: same request always goes to same model."""
        active = [v for v in self.versions.values() if v.status in ('canary', 'production')]
        bucket = int(hashlib.md5(request_id.encode()).hexdigest()[:4], 16) % 1000 / 1000
        cumulative = 0.0
        for v in sorted(active, key=lambda x: x.version_id):
            cumulative += v.traffic_pct
            if bucket < cumulative:
                return v
        return active[-1]

    def promote_canary(self, version_id: str, new_pct: float):
        canary = self.versions[version_id]
        prod = next(v for v in self.versions.values() if v.status == 'production')
        delta = new_pct - canary.traffic_pct
        canary.traffic_pct = new_pct
        prod.traffic_pct = max(0, prod.traffic_pct - delta)
        if new_pct >= 1.0:
            canary.status = 'production'
            prod.status = 'retired'
        print(f'📈 Promoted {version_id} to {new_pct:.0%} canary (prod now {prod.traffic_pct:.0%})')

    def rollback(self, reason: str):
        for v in self.versions.values():
            if v.status == 'canary':
                v.traffic_pct = 0.0
                v.status = 'shadow'
            elif v.status == 'production':
                v.traffic_pct = 1.0
        print(f'🚨 ROLLBACK: {reason}')

    def status_report(self):
        print('\n=== Model Registry Status ===')
        for v in self.versions.values():
            acc_str = f'{v.accuracy:.1%}' if v.accuracy else 'N/A'
            lat_str = f'{v.p50_ms:.0f}ms' if v.p50_ms else 'N/A'
            print(f'  {v.version_id:<20} [{v.status:<12}] traffic={v.traffic_pct:.0%}  acc={acc_str}  p50={lat_str}')


# Simulate 4-week deployment lifecycle
reg = ModelRegistry()
reg.register(ModelVersion('v3.5-base',      'claude-sonnet', 'production', 1.0,  accuracy=0.91, p50_ms=850))
reg.register(ModelVersion('v3.6-ft-shadow', 'loanbot-ft',   'shadow',     0.0,  accuracy=None, p50_ms=None))

print('\n--- Week 1-2: Shadow mode monitoring ---')
reg.versions['v3.6-ft-shadow'].accuracy = 0.972
reg.versions['v3.6-ft-shadow'].p50_ms = 140
reg.status_report()

print('\n--- Week 3: Promote to canary 10% ---')
reg.versions['v3.6-ft-shadow'].status = 'canary'
reg.versions['v3.6-ft-shadow'].version_id = 'v3.6-ft-canary'
reg.versions['v3.6-ft-canary'] = reg.versions.pop('v3.6-ft-shadow')
reg.promote_canary('v3.6-ft-canary', 0.10)

print('\n--- Week 4: All good, promote to 50% ---')
reg.promote_canary('v3.6-ft-canary', 0.50)
reg.status_report()

print('\n--- Simulate incident: rollback ---')
reg.rollback('GPU temperature spike → 100% REJECT anomaly in batch')
reg.status_report()

# Test routing
print('\n--- Routing test (100 requests) ---')
# Reset to 10% canary for routing test
reg.versions['v3.5-base'].traffic_pct = 1.0
route_counts = {}
for i in range(100):
    v = reg.route(f'REQ-{i:04d}')
    route_counts[v.version_id] = route_counts.get(v.version_id, 0) + 1
for k, c in route_counts.items():
    print(f'  {k}: {c} requests ({c}%)')

## Section 8: Practice — Complete Pipeline Orchestrator

**Bài tập:** Hoàn thiện `FineTuningPipeline` — một class tích hợp tất cả components từ sections trên.

TODO sections yêu cầu bạn implement logic dựa trên kiến thức từ module.

In [ ]:
class FineTuningPipeline:
    """Complete pipeline: ROI → Data → Train → Evaluate → Deploy."""

    def __init__(self, config: LoRAConfig, roi: FineTuningROI):
        self.config = config
        self.roi = roi
        self.training_data: List[TrainingExample] = []
        self.best_model = None
        self.registry = ModelRegistry()

    def phase1_roi_check(self) -> bool:
        """TODO: Return True only if ROI is positive AND break-even < model_lifespan."""
        # YOUR CODE HERE
        pass

    def phase2_prepare_data(self, raw_cases: List[Tuple]) -> int:
        """TODO: Anonymize + generate + filter examples. Return count of filtered examples."""
        # Hint: use anonymize(), mock_generate_example(), filter_examples()
        # YOUR CODE HERE
        pass

    def phase3_train(self, n_epochs: int = 5) -> Dict:
        """TODO: Simulate training with early stopping. Return best checkpoint info."""
        # Hint: use simulate_training() and EarlyStopping
        # YOUR CODE HERE
        pass

    def phase4_evaluate(self, model) -> bool:
        """TODO: Run 4-dimension evaluation. Return True only if ALL dimensions pass."""
        # Hint: domain_acc >= 0.95, json_compliance >= 0.99, general_capability >= 0.85
        # YOUR CODE HERE
        pass

    def phase5_deploy_shadow(self):
        """TODO: Register base model (production 100%) and new model (shadow 0%)."""
        # YOUR CODE HERE
        pass

    def run(self, raw_cases: List[Tuple]) -> str:
        print('🚀 Starting LoanBot Fine-tuning Pipeline\n')

        if not self.phase1_roi_check():
            return '❌ Stopped: ROI check failed'

        n = self.phase2_prepare_data(raw_cases)
        if n is None or n < 100:
            return f'❌ Stopped: insufficient training data ({n} examples)'

        checkpoint = self.phase3_train()
        if checkpoint is None:
            return '❌ Stopped: training failed'

        ft_model = MockFineTunedModel()
        if not self.phase4_evaluate(ft_model):
            return '❌ Stopped: evaluation failed'

        self.phase5_deploy_shadow()
        return '✅ Pipeline complete — model in shadow mode, monitoring begins'


# --- SOLUTION (hidden by default) ---
class FineTuningPipelineSolution(FineTuningPipeline):

    def phase1_roi_check(self) -> bool:
        ok = self.roi.total_roi() > 0 and self.roi.break_even_months() < self.roi.model_lifespan_months
        print(f'Phase 1 ROI: {self.roi.recommend()}')
        return ok

    def phase2_prepare_data(self, raw_cases) -> int:
        raw = [mock_generate_example(app, dec) for app, dec in raw_cases]
        self.training_data = filter_examples(raw, threshold=0.70)
        print(f'Phase 2 Data: {len(raw)} raw → {len(self.training_data)} filtered')
        return len(self.training_data)

    def phase3_train(self, n_epochs=5) -> Dict:
        history = simulate_training(n_epochs, len(self.training_data), self.config.r)
        stopper = EarlyStopping(patience=2)
        best = None
        for h in history:
            if stopper.step(h['epoch'], h['val_acc']) or h == history[-1]:
                best = {'epoch': stopper.best_epoch, 'val_acc': stopper.best_val_acc}
                break
        print(f'Phase 3 Train: best checkpoint epoch={best["epoch"]} val_acc={best["val_acc"]:.4f}')
        return best

    def phase4_evaluate(self, model) -> bool:
        correct = sum(1 for app in TEST_APPS if model.predict(app)['decision'] == app['expert'])
        acc = correct / len(TEST_APPS)
        valid = sum(1 for app in TEST_APPS if {'decision','confidence','explanation'}.issubset(model.predict(app)))
        jc = valid / len(TEST_APPS)
        gc = 1.0  # mock fine-tuned model passes general tasks
        passed = acc >= 0.90 and jc >= 0.99 and gc >= 0.85
        print(f'Phase 4 Eval: domain={acc:.1%} json={jc:.1%} general={gc:.1%} → {"PASS" if passed else "FAIL"}')
        return passed

    def phase5_deploy_shadow(self):
        self.registry.register(ModelVersion('v3.5-base', 'claude-sonnet', 'production', 1.0, 0.91, 850))
        self.registry.register(ModelVersion('v3.6-ft',   'loanbot-ft',   'shadow',     0.0, None, None))
        print('Phase 5 Deploy: v3.6-ft registered in shadow mode')


# Run solution
cfg = LoRAConfig(r=16, alpha=32, target_modules=['q_proj','v_proj','k_proj','o_proj'],
                 dropout=0.05, base_model_params_B=7.0, hidden_dim=4096)
roi = FineTuningROI(50_000, 0.008, 0.002, 2_000, 18)

pipeline = FineTuningPipelineSolution(cfg, roi)
result = pipeline.run([random_case(i) for i in range(200)])
print(f'\n🏁 Result: {result}')